# Data Gathering

Pada tahap data gathering, dilakukan proses pengambilan dataset pekerjaan menggunakan library Pandas. Dataset dibaca dari file CSV bernama `Dataset_pekerjaan.csv` untuk selanjutnya diproses pada tahap analisis dan pembersihan data.

In [65]:
import pandas as pd
import re
!pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


df = pd.read_csv("Dataset_pekerjaan.csv")
df.head()

,Jabatan,Kode Unit,Judul Unit,Elemen Kompetensi,Kriteria Unjuk Kerja (KUK),Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Manager,C.29TDI01.001.1,Merumuskan Aspirasi Transformasi Industri 4.0,Mengenali industri 4.0,"1.1 Latar belakang, keuntungan dan tantangan i...",NaN,NaN,NaN,NaN,NaN
1,Manager,C.29TDI01.001.1,Merumuskan Aspirasi Transformasi Industri 4.0,Mengidentifikasi core element industri 4.0,2.1 Core technology untuk industri 4.0 diident...,NaN,NaN,NaN,NaN,NaN
2,Manager,C.29TDI01.001.1,Merumuskan Aspirasi Transformasi Industri 4.0,Membuat peta pemangku kepentingan,3.1 Pemangku kepentingan terkait transformasi ...,NaN,NaN,NaN,NaN,NaN
3,Manager,C.29TDI01.002.1,Merumuskan Peluang Penerapan Transformasi Indu...,Melakukan business model analysis dan business...,1.1 Business model yang sedang berjalan diiden...,NaN,NaN,NaN,NaN,NaN
4,Manager,C.29TDI01.002.1,Merumuskan Peluang Penerapan Transformasi Indu...,Melakukan gap analysis,2.1 Hasil analisis kebutuhan dibandingkan deng...,NaN,NaN,NaN,NaN,NaN


# Data Assessing

Tahap assessing data dilakukan untuk memahami struktur dan kualitas dataset.
Beberapa proses yang dilakukan antara lain:

- Menghapus kolom yang tidak digunakan (`Unnamed`)
- Melihat sampel data menggunakan `head()`
- Mengecek tipe data dan jumlah data dengan `info()`
- Mengecek missing values menggunakan `isnull().sum()`

Tahap ini bertujuan untuk mengidentifikasi permasalahan data sebelum masuk ke proses cleaning.

In [66]:
df = df.drop(columns=['Unnamed: 5','Unnamed: 6','Unnamed: 7','Unnamed: 8','Unnamed: 9'])

df.info()

df.head()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683 entries, 0 to 682
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Jabatan                     673 non-null    object
 1   Kode Unit                   673 non-null    object
 2   Judul Unit                  673 non-null    object
 3   Elemen Kompetensi           673 non-null    object
 4   Kriteria Unjuk Kerja (KUK)  673 non-null    object
dtypes: object(5)
memory usage: 26.8+ KB


,0
Jabatan,10
Kode Unit,10
Judul Unit,10
Elemen Kompetensi,10
Kriteria Unjuk Kerja (KUK),10


# Data Cleaning

Pada tahap data cleaning dilakukan beberapa proses pembersihan data agar dataset lebih terstruktur dan siap digunakan pada tahap analisis maupun pemodelan.

Tahapan cleaning yang dilakukan meliputi:

1. Menghapus baris yang seluruh nilainya kosong
2. Membersihkan nomor pada teks KUK menggunakan regular expression
3. Memisahkan isi KUK berdasarkan titik dan titik koma
4. Mengubah setiap hasil split menjadi baris baru menggunakan `explode()`
5. Menghapus kolom lama yang belum dibersihkan
6. Melakukan case folding (mengubah teks menjadi huruf kecil)
7. Menghapus spasi berlebih
8. Menghapus karakter khusus yang tidak diperlukan
9. Melakukan stopword removal menggunakan library Sastrawi

Hasil cleaning menghasilkan dataset yang lebih rapi dan konsisten

In [67]:
df = df.dropna(how='all')

def split_kuk(text):
    text = re.sub(r'\b\d+(\.\d+)+\b', '', text)

    parts = re.split(r';|\.\s+(?=[a-zA-Z])', text)

    return [p.strip() for p in parts if p.strip()]

df['KUK_list'] = df['Kriteria Unjuk Kerja (KUK)'].apply(split_kuk)

df = df.explode('KUK_list')

df['KUK'] = df['KUK_list']

df = df.drop(columns=['KUK_list', 'Kriteria Unjuk Kerja (KUK)'])

df['KUK'] = df['KUK'].str.lower()

df['KUK'] = df['KUK'].apply(lambda x: re.sub(r'\s+', ' ', x))

df['KUK'] = df['KUK'].str.strip()

df['KUK'] = df['KUK'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s,\.]', '', x)
)

# stopword removal
factory = StopWordRemoverFactory()
stopword = factory.create_stop_word_remover()

df['KUK'] = df['KUK'].apply(lambda x: stopword.remove(x))

## Save dataset
Menyimpan hasil akhir dataset cleaning ke file CSV baru

In [68]:
df.to_csv("data_pekerjaan.csv", index=False)